In [1]:
import re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime as dt
from sklearn import set_config
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer, PowerTransformer, StandardScaler 
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import pickle

ERROR! Session/line number was not unique in database. History logging moved to new session 209


In [2]:
df = pd.read_csv("datasets/car_details.csv")
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [ ]:
df.isnull().sum()

In [30]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['selling_price'], axis=1), 
                                                    df['selling_price'],
                                                    test_size=0.2,
                                                    random_state=0)
X_train.shape, X_test.shape

((6502, 12), (1626, 12))

In [31]:
def extract_name(df):
    df = df.copy()
    df['brand'] = df['name'].str.split().str[0]
    df['car_model'] = df['name'].str.split().str[1]
    df.drop(columns=['name'], axis = 1, inplace=True)
    return df

name_transformer = FunctionTransformer(extract_name)

In [32]:
def extract_car_age(df):
    df = df.copy()
    df['car_age'] = dt.now().year - df['year']
    df.drop(columns=['year'], axis = 1, inplace=True)
    return df

year_transformer = FunctionTransformer(extract_car_age)

In [33]:
def extract_mileage(df):
    df = df.copy()
    df['mileage'] = df['mileage'].str.extract(r'(\d+\.?\d*)').astype(np.float64)
    return df

mileage_transformer = FunctionTransformer(extract_mileage)

In [34]:
def extract_engine (df):
    df = df.copy()
    df['engine'] = df['engine'].str.extract(r'(\d+\.?\d*)').astype(np.float64)
    return df

engine_transformer = FunctionTransformer(extract_engine)

In [35]:
def extract_max_power(df):
    df = df.copy()
    df['max_power'] = df['max_power'].str.extract(r'(\d+\.?\d*)').astype(np.float64)
    return df

max_power_transformer = FunctionTransformer(extract_max_power)

In [36]:
def clean_torque(df):
    df = df.copy()

    def extract_torque(val):
        if pd.isna(val):
            return np.nan
        val = str(val).replace(',', '') 
        match = re.search(r'(\d+\.?\d*)\s*(Nm|kgm)', val, re.IGNORECASE)
        if not match:
            return np.nan
        number, unit = float(match.group(1)), match.group(2).lower()
        if unit == 'kgm':
            number = number * 9.8
        return number

    df['torque'] = df['torque'].apply(extract_torque)
    return df

torque_transformer = FunctionTransformer(clean_torque)

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB


In [38]:
cleaning_pipe = Pipeline(steps = [
    ('name', name_transformer),
    ('year', year_transformer),
    ('mileage', mileage_transformer),
    ('engine', engine_transformer),
    ('max_power', max_power_transformer),
    ('torque', torque_transformer)
])

In [ ]:
df_clean = cleaning_pipe.fit_transform(df)
df_clean.skew(numeric_only=True)

In [ ]:
df_clean['brand'].value_counts()

In [39]:
iterative_imputer = IterativeImputer(estimator=RandomForestRegressor(n_estimators=10, random_state=0), max_iter=50, random_state= 0) 

categorial_imputer_pipe = Pipeline(steps = [
    ('impute', SimpleImputer(strategy='most_frequent')),
])

One_Hot_pipe = Pipeline(steps = [
    ('OHE', OneHotEncoder(handle_unknown='ignore',sparse_output=False, dtype = np.int64, drop='first'))
])



owner_pipe = Pipeline(steps=[
    ('encoding', OrdinalEncoder(categories=[['Fourth & Above Owner', 
                                             'Third Owner', 
                                             'Second Owner', 
                                             'First Owner',
                                             'Test Drive Car']], dtype = np.int64))
])

car_model_pipe = Pipeline(steps = [
    ('encode', OneHotEncoder(min_frequency=50, handle_unknown='infrequent_if_exist', dtype = int, sparse_output=False, drop='first'))
])

brand_pipe = Pipeline(steps=[
    ('encode', OneHotEncoder(min_frequency=30, sparse_output=False, handle_unknown='infrequent_if_exist', dtype=int))
])

positive_skewness_pipe = Pipeline(steps=[
    ('log', FunctionTransformer(func=np.log1p)),
    ('scale', StandardScaler())
])

negative_skewness_pipe = Pipeline(steps = [
    ('square', FunctionTransformer(func=np.square)),
    ('scale', StandardScaler())
])

heavy_positive_skewness_pipe = Pipeline(steps = [
    ('yeo-johnson', PowerTransformer(method='yeo-johnson'))
])

scaler_pipe = Pipeline(steps = [
    ('scale', StandardScaler())
])

imputer = ColumnTransformer(transformers=[
    ('categorial_imputer', categorial_imputer_pipe, ['seats']),
    ('iterative_imputer', iterative_imputer, ['mileage', 'car_age', 'torque', 'max_power', 'engine'])
], remainder='passthrough', verbose_feature_names_out=False)

In [ ]:
df_imputed = pd.DataFrame(imputer.fit_transform(df_clean), columns= imputer.get_feature_names_out())

In [40]:
transform = ColumnTransformer(transformers = [
    ('owner', owner_pipe, ['owner']),
    ('ohe', One_Hot_pipe, ['fuel', 'seller_type', 'transmission']),
    ('car_model', car_model_pipe, ['car_model']),
    ('brand', brand_pipe, ['brand']),
    ('positive_skewness', positive_skewness_pipe, ['engine', 'torque', 'car_age']),
    ('heavy_positive', heavy_positive_skewness_pipe, ['km_driven']),
    ('negative_skewness', negative_skewness_pipe, ['max_power']),
    ('scale', scaler_pipe, ['mileage'])
], remainder='passthrough')

In [41]:
def to_dataframe(arr):
    dataframe = pd.DataFrame(arr, columns= imputer.get_feature_names_out())
    dataframe = dataframe.infer_objects()
    return dataframe

In [42]:
preprocessor = Pipeline(steps=[
    ('clean', cleaning_pipe),
    ('imputer', imputer),
    ('dataframe', FunctionTransformer(func= to_dataframe)),
    ('transform', transform)
])

In [43]:
arr = preprocessor.named_steps['imputer'].fit_transform(df_clean)

type(arr), arr.shape

KeyboardInterrupt: 

In [ ]:
dtfm = preprocessor.named_steps['dataframe'].fit_transform(arr)

dtfm.info()

In [ ]:
preprocessor.named_steps['transform'].fit(dtfm)

In [44]:
final_pipe = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=0))
])

In [47]:
final_pipe.fit(X_train, y_train)

# pred = final_pipe.predict(X_test)

# r2_score(y_test, pred)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('clean', ...), ('imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('name', ...), ('year', ...), ...]"
,"transform_input t

In [48]:
cross_val_score(final_pipe, X=df.drop(columns = ['selling_price'], axis = 1), y = df['selling_price'], cv=5, scoring='r2')

C:\Users\Keval\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
C:\Users\Keval\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
C:\Users\Keval\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
C:\Users\Keval\AppData\Loca

array([0.96438731, 0.97209199, 0.97707906, 0.95528155, 0.96746831])

In [ ]:
# for col in df_transformed.drop(columns=['selling_price'], axis = 1).columns:
#     sns.scatterplot(x = df_transformed[col],y = df_transformed['selling_price'])
#     plt.show()

Encode - fuel, seller_type, transmission, owner, brand, car_model

scale - selling_price, km_driven, mileange, engine, max_power


In [ ]:
for col in df.columns:
    for col2 in df.columns:
        print(f"{col} with {col2} : {df.groupby(col)[col2].apply(lambda x: x.isnull().mean())}")

In [ ]:
for col in df.columns:
    print(col, "->", df[col].dtype, "| unique values:", df[col].nunique())

In [ ]:
df.isnull().mean() * 100

In [ ]:
# Is a column more likely to be missing depending on another column's value?
df.groupby('owner')['mileage'].apply(lambda x: x.isnull().mean())

In [ ]:
for col in df_clean.select_dtypes(include='number').columns:
    print(f"{col} and Selling Price : {df_clean[[col, 'selling_price']].corr()[col]['selling_price']}")

In [ ]:
for col in df_clean.select_dtypes(include='object').columns:
    print(df_clean.groupby(col)['selling_price'].median().sort_values())

In [ ]:
sns.heatmap(df_clean.corr(numeric_only=True), annot=True, cmap='coolwarm')

In [ ]:
np.where()

In [51]:
pickle.dump(final_pipe, open('models/Car_price_prediction.pkl', 'wb'))

In [52]:
model = pickle.load(open('models/Car_price_prediction.pkl', 'rb'))

In [53]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('clean', ...), ('imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('name', ...), ('year', ...), ...]"
,"transform_input t